What it does: Takes ANY CSV file as input, automatically detects numeric columns, generates statistics, creates histograms, and saves a self-contained HTML report (images embedded as base64).

In [1]:
# Cell 1: Import all required libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import base64
import io
from pathlib import Path

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [2]:
# Cell 2: Function to load CSV and auto-detect numeric columns
def load_and_detect_csv(filepath):
    """Load CSV and identify numeric columns"""
    try:
        df = pd.read_csv(filepath)
        print(f"✅ Loaded '{filepath}'")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        
        # Detect numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        
        print(f"   Numeric columns ({len(numeric_cols)}): {numeric_cols}")
        print(f"   Categorical columns ({len(categorical_cols)}): {categorical_cols}")
        
        return df, numeric_cols, categorical_cols
    except FileNotFoundError:
        print(f"❌ Error: File '{filepath}' not found!")
        return None, [], []
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None, [], []

In [3]:
# Cell 3: Function to generate summary statistics
def generate_statistics(df, numeric_cols):
    """Generate summary statistics for numeric columns"""
    stats = df[numeric_cols].describe().round(2)
    
    # Add missing values count
    missing = df[numeric_cols].isnull().sum()
    stats.loc['missing'] = missing
    
    # Add skewness (measure of distribution asymmetry)
    skewness = df[numeric_cols].skew().round(3)
    stats.loc['skewness'] = skewness
    
    return stats

In [7]:
# Cell 4: Function to create histograms and return as base64
def create_histograms_base64(df, numeric_cols):
    """Create histograms for all numeric columns and return as base64 strings"""
    n_cols = len(numeric_cols)
    if n_cols == 0:
        return None
    
    # Calculate grid size (max 3 columns)
    n_rows = (n_cols + 2) // 3
    fig, axes = plt.subplots(n_rows, min(3, n_cols), figsize=(15, 5*n_rows))
    
    # Handle single subplot case
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        ax = axes[i]
        
        # Plot histogram
        ax.hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.7)
        
        # Add vertical lines for mean and median
        ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {df[col].mean():.2f}')
        ax.axvline(df[col].median(), color='green', linestyle=':', linewidth=2, 
                   label=f'Median: {df[col].median():.2f}')
        
        # Add labels and title
        ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for i in range(len(numeric_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    
    # Convert to base64
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close()
    buf.close()
    
    return img_base64

In [8]:
# Cell 5: Create a sample CSV file for testing (if none provided)
def create_sample_csv():
    """Create a sample sensor CSV file for demonstration"""
    np.random.seed(42)
    n_samples = 1000
    
    sample_data = pd.DataFrame({
        'timestamp': pd.date_range('2025-05-01', periods=n_samples, freq='5min'),
        'temperature_c': np.random.normal(45, 10, n_samples),
        'vibration_mm_s': np.random.normal(2.5, 0.8, n_samples),
        'current_a': np.random.normal(5, 1.5, n_samples),
        'power_w': np.random.normal(1100, 300, n_samples),
        'status': np.random.choice(['NORMAL', 'WARNING', 'ALERT'], n_samples, p=[0.85, 0.10, 0.05])
    })
    
    sample_data.to_csv('sample_sensor_data.csv', index=False)
    print("✅ Created sample file: 'sample_sensor_data.csv'")
    return 'sample_sensor_data.csv'

In [10]:
# Cell 6: Generate complete HTML report with embedded images
def generate_html_report(df, stats, img_base64, filename, numeric_cols):
    """Generate self-contained HTML report with base64 embedded images"""
    
    # Convert statistics to HTML table
    stats_html = stats.to_html() if stats is not None else "<p>No numeric columns found</p>"
    
    # Create HTML content
    html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>CSV Explorer Report: {filename}</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            margin: 40px;
            background-color: #f5f5f5;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 30px;
            border-left: 4px solid #3498db;
            padding-left: 15px;
        }}
        .summary-box {{
            background-color: #e8f4f8;
            padding: 20px;
            border-radius: 8px;
            margin: 20px 0;
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
        }}
        .summary-card {{
            background: white;
            padding: 15px;
            border-radius: 5px;
            text-align: center;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        .summary-card h3 {{
            margin: 0 0 10px 0;
            color: #3498db;
            font-size: 14px;
        }}
        .summary-card .value {{
            font-size: 24px;
            font-weight: bold;
            color: #2c3e50;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 20px 0;
            font-size: 14px;
        }}
        th, td {{
            border: 1px solid #ddd;
            padding: 10px;
            text-align: center;
        }}
        th {{
            background-color: #3498db;
            color: white;
            font-weight: bold;
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        .histogram {{
            text-align: center;
            margin: 30px 0;
        }}
        .histogram img {{
            max-width: 100%;
            border: 1px solid #ddd;
            border-radius: 8px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }}
        .footer {{
            text-align: center;
            margin-top: 40px;
            padding-top: 20px;
            border-top: 1px solid #ddd;
            color: #7f8c8d;
            font-size: 12px;
        }}
        .insight {{
            background-color: #d4edda;
            border-left: 4px solid #28a745;
            padding: 15px;
            margin: 20px 0;
            border-radius: 5px;
        }}
        .warning {{
            background-color: #fff3cd;
            border-left: 4px solid #ffc107;
            padding: 15px;
            margin: 20px 0;
            border-radius: 5px;
        }}
    </style>
</head>
<body>
<div class="container">
    <h1>📊 CSV Explorer Report</h1>
    <p><strong>File:</strong> {filename}</p>
    <p><strong>Generated:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    
    <div class="summary-box">
        <div class="summary-card">
            <h3>Total Rows</h3>
            <div class="value">{len(df):,}</div>
        </div>
        <div class="summary-card">
            <h3>Total Columns</h3>
            <div class="value">{len(df.columns)}</div>
        </div>
        <div class="summary-card">
            <h3>Numeric Columns</h3>
            <div class="value">{len(numeric_cols)}</div>
        </div>
        <div class="summary-card">
            <h3>Missing Values</h3>
            <div class="value">{df.isnull().sum().sum():,}</div>
        </div>
    </div>
"""
    
    # Add insights if there are numeric columns
    if len(numeric_cols) > 0:
        html_content += f"""
    <div class="insight">
        <strong>💡 Key Insights:</strong><br>
        • Temperature range: {df[numeric_cols[0]].min():.2f} to {df[numeric_cols[0]].max():.2f}<br>
        • Data appears to be {len(df)} readings over time<br>
        • Consider checking for outliers in the histogram below
    </div>
"""
    
    # Add histogram
    if img_base64:
        html_content += f"""
    <h2>📈 Distribution of Numeric Columns</h2>
    <div class="histogram">
        <img src="data:image/png;base64,{img_base64}" alt="Histograms">
    </div>
"""
    
    # Add statistics table
    if stats is not None:
        html_content += f"""
    <h2>📋 Summary Statistics</h2>
    {stats_html}
"""
    
    # Add column information
    html_content += f"""
    <h2>📝 Column Information</h2>
    <table>
        <tr><th>Column Name</th><th>Data Type</th><th>Non-Null Count</th><th>Unique Values</th></tr>
"""
    for col in df.columns:
        dtype = df[col].dtype
        non_null = df[col].count()
        unique = df[col].nunique()
        html_content += f"        <tr><td>{col}</td><td>{dtype}</td><td>{non_null:,}</td><td>{unique:,}</td></tr>\n"
    
    html_content += f"""
    </table>
    
    <div class="footer">
        <p>Report generated by SEED AI & ML for Electrical Engineers - Automated CSV Explorer</p>
        <p>This report contains embedded visualizations and does not require external files.</p>
    </div>
</div>
</body>
</html>
"""
    
    return html_content

In [11]:
# Cell 7: Main function to run the CSV Explorer
def run_csv_explorer(csv_path=None):
    """Main function to run the CSV explorer"""
    
    print("="*60)
    print("🤖 AUTOMATED CSV EXPLORER")
    print("="*60)
    
    # If no file provided, use sample or ask user
    if csv_path is None:
        try:
            # Try to use provided command line argument
            if len(sys.argv) > 1:
                csv_path = sys.argv[1]
            else:
                # Create sample file for demonstration
                csv_path = create_sample_csv()
        except:
            csv_path = create_sample_csv()
    
    # Load and detect CSV
    df, numeric_cols, categorical_cols = load_and_detect_csv(csv_path)
    
    if df is None:
        print("❌ Failed to load CSV. Exiting.")
        return
    
    # Generate statistics
    if len(numeric_cols) > 0:
        stats = generate_statistics(df, numeric_cols)
        print("\n📊 Statistics generated successfully")
        
        # Create histograms
        print("🎨 Creating histograms...")
        img_base64 = create_histograms_base64(df, numeric_cols)
        print("   ✓ Histograms created and encoded")
    else:
        stats = None
        img_base64 = None
        print("⚠️ No numeric columns found - skipping histograms")
    
    # Generate HTML report
    print("📄 Generating HTML report...")
    html_content = generate_html_report(df, stats, img_base64, csv_path, numeric_cols)
    
    # Save report
    output_filename = f"report_{Path(csv_path).stem}.html"
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✅ Report saved as '{output_filename}'")
    print(f"\n📁 Output files:")
    print(f"   - {output_filename} (HTML report)")
    
    # Print quick summary
    print("\n" + "="*60)
    print("📊 QUICK SUMMARY")
    print("="*60)
    print(f"Total rows:     {len(df):,}")
    print(f"Total columns:  {len(df.columns)}")
    print(f"Numeric cols:   {len(numeric_cols)}")
    print(f"Categorical:    {len(categorical_cols)}")
    print(f"Missing values: {df.isnull().sum().sum():,}")
    
    if len(numeric_cols) > 0:
        print(f"\nTop numeric columns: {numeric_cols[:5]}")
    
    return output_filename

In [12]:
# Cell 8: Execute the CSV explorer
# Option 1: Create and analyze sample data
report_file = run_csv_explorer()

# Option 2: Analyze your own CSV file (uncomment and modify path)
# report_file = run_csv_explorer('your_data.csv')

print("\n🎉 CSV Explorer completed successfully!")
print("   Open the HTML file in your browser to view the report.")

🤖 AUTOMATED CSV EXPLORER
❌ Error: File '-f' not found!
❌ Failed to load CSV. Exiting.

🎉 CSV Explorer completed successfully!
   Open the HTML file in your browser to view the report.


In [13]:
# Cell 9: Test with different scenarios

# Test 1: Create another sample file with different data
print("\n" + "="*60)
print("🧪 TESTING WITH DIFFERENT DATA TYPES")
print("="*60)

# Create test file with missing values
test_data = pd.DataFrame({
    'id': range(1, 101),
    'value1': np.random.normal(100, 15, 100),
    'value2': np.random.uniform(0, 1, 100),
    'value3': np.random.exponential(10, 100),
    'category': np.random.choice(['A', 'B', 'C'], 100)
})

# Introduce some missing values
test_data.loc[10:15, 'value1'] = np.nan
test_data.loc[30:35, 'value2'] = np.nan

test_data.to_csv('test_missing_data.csv', index=False)
print("✅ Created 'test_missing_data.csv' with missing values")

# Analyze it
run_csv_explorer('test_missing_data.csv')


🧪 TESTING WITH DIFFERENT DATA TYPES
✅ Created 'test_missing_data.csv' with missing values
🤖 AUTOMATED CSV EXPLORER
✅ Loaded 'test_missing_data.csv'
   Shape: 100 rows × 5 columns
   Numeric columns (4): ['id', 'value1', 'value2', 'value3']
   Categorical columns (1): ['category']

📊 Statistics generated successfully
🎨 Creating histograms...
   ✓ Histograms created and encoded
📄 Generating HTML report...
✅ Report saved as 'report_test_missing_data.html'

📁 Output files:
   - report_test_missing_data.html (HTML report)

📊 QUICK SUMMARY
Total rows:     100
Total columns:  5
Numeric cols:   4
Categorical:    1
Missing values: 12

Top numeric columns: ['id', 'value1', 'value2', 'value3']


'report_test_missing_data.html'

In [14]:
# Cell 10: Instructions for command-line usage
print("""
╔══════════════════════════════════════════════════════════════╗
║                    COMMAND LINE USAGE                        ║
╠══════════════════════════════════════════════════════════════╣
║  To run from command line:                                  ║
║                                                             ║
║    python csv_explorer.py your_data.csv                     ║
║                                                             ║
║  Or run this notebook and it will:                          ║
║    1. Create sample data if no file provided                ║
║    2. Analyze any CSV you point it to                       ║
║    3. Generate HTML report with embedded histograms         ║
║                                                             ║
║  Features:                                                  ║
║    ✓ Auto-detects numeric vs categorical columns            ║
║    ✓ Handles missing values gracefully                      ║
║    ✓ Creates histograms for all numeric columns             ║
║    ✓ Saves self-contained HTML report                       ║
║    ✓ No external files needed - images embedded as base64   ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║                    COMMAND LINE USAGE                        ║
╠══════════════════════════════════════════════════════════════╣
║  To run from command line:                                  ║
║                                                             ║
║    python csv_explorer.py your_data.csv                     ║
║                                                             ║
║  Or run this notebook and it will:                          ║
║    1. Create sample data if no file provided                ║
║    2. Analyze any CSV you point it to                       ║
║    3. Generate HTML report with embedded histograms         ║
║                                                             ║
║  Features:                                                  ║
║    ✓ Auto-detects numeric vs categorical columns            ║
║    ✓ Handles missing values gracefully                      ║
║    ✓ Creates histograms for all nu